# Pipeline RAG untuk Generate Literature Gap Matrix & Analisis Usulan Paper (S2 AI)
**Domain Fokus:** Edukasi & Kesehatan Remaja terkait Kecanduan Smartphone (*Smartphone Addiction*) dan Pencegahan Merokok/Vaping (*Smoking/E-Cigarette Cessation*).

Notebook ini menjalankan keseluruhan tahapan **Retrieval-Augmented Generation (RAG)**:
1. **PDF Ingestion & Recursive Chunking** dengan Metadata Tagging (`filename`, `page_number`).
2. **Hybrid Vector Indexing** (Google Gemini Embeddings `gemini-embedding-001` + Lexical TF-IDF Vectorizer).
3. **Per-Paper Multi-Aspect Retrieval & Structured Gap Extraction** (`gemini-2.5-flash` + Pydantic Schema).
4. **Cross-Paper Synthesis** untuk menghasilkan **Analisis Paragraf & Rekomendasi Usulan Paper Kelompok**.

In [ ]:
import sys
from pathlib import Path

# Tambahkan root project ke sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PAPERS_DIR, OUTPUTS_DIR, VECTOR_STORE_DIR, CHUNK_SIZE, CHUNK_OVERLAP, DOMAIN_TOPIC
from src.sample_papers_generator import ensure_sample_papers_if_empty
from src.pdf_loader import load_all_papers
from src.vector_store import AcademicVectorStore
from src.gap_extractor import LiteratureGapRAGPipeline
from src.report_exporter import export_all_outputs

print('Domain Fokus:', DOMAIN_TOPIC)

In [ ]:
# Langkah 1 & 2: Load PDF dari data/papers/ dan lakukan Chunking
ensure_sample_papers_if_empty(PAPERS_DIR)
papers_dict = load_all_papers(PAPERS_DIR, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
all_chunks = [c for chunks in papers_dict.values() for c in chunks]
print(f'Total Paper PDF: {len(papers_dict)} | Total Chunks: {len(all_chunks)}')

In [ ]:
# Langkah 3: Indexing ke Vector Store & Ekstraksi Literature Gap Matrix
vector_store = AcademicVectorStore(persist_dir=VECTOR_STORE_DIR)
vector_store.index_chunks(all_chunks)

pipeline = LiteratureGapRAGPipeline(vector_store=vector_store)
gap_rows = []
for idx, (filename, p_chunks) in enumerate(papers_dict.items(), start=1):
    row = pipeline.extract_gap_row(idx=idx, filename=filename, paper_chunks=p_chunks)
    gap_rows.append(row)
    print(f'[{idx}] Berhasil mengekstrak: {row.penulis_tahun} - {row.publisher_jurnal}')

In [ ]:
# Langkah 4: Generate Sintesis Analisis Paragraf & Simpan ke File
synthesis = pipeline.generate_synthesis_analysis(gap_rows)
exported = export_all_outputs(outputs_dir=OUTPUTS_DIR, gap_rows=gap_rows, synthesis=synthesis)

print('Usulan Judul Paper Kelompok:\n', synthesis.usulan_judul_paper)
print('\nFile Laporan Markdown:', exported['markdown'])